In [1]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import collections

In [2]:
def get_age_class(age):
    if age < 18:
        return 0   # Young
    elif age < 40:
        return 1   # Adult
    else:
        return 2   # Old

In [3]:
age_groups = ["Young", "Adult", "Old"]
age_ranges = ["0-17 yr", "18-39 yr", "40+ yr"]

In [4]:
data_dir = "dataset/UTKFace"

images = []
labels = []

for file in os.listdir(data_dir):
    try:
        age = int(file.split("_")[0])
        img_path = os.path.join(data_dir, file)

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.resize(img, (64, 64))

        images.append(img)
        labels.append(get_age_class(age))

    except:
        continue

In [5]:
images = np.array(images, dtype="float32") / 255.0
labels = np.array(labels)

In [6]:
print("Class Distribution:", collections.Counter(labels))

Class Distribution: Counter({np.int64(1): 12241, np.int64(2): 7234, np.int64(0): 4233})


In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

In [8]:
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: np.float64(1.8670998227997637), 1: np.float64(0.6455631573572961), 2: np.float64(1.0924485916709867)}


In [9]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
])

In [10]:
model = tf.keras.models.Sequential([
    layers.Input(shape=(64, 64, 3)),
    data_augmentation,

    # Conv Block 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    # Conv Block 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    # Conv Block 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    # Dense
    layers.Dense(128, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    # OUTPUT (3 classes)
    layers.Dense(3, activation='softmax')
])

In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [12]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [13]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/30
593/593 ━━━━━━━━━━━━━━━━━━━━ 156s 251ms/step - accuracy: 0.5373 - loss: 1.2420 - val_accuracy: 0.7153 - val_loss: 1.0208
Epoch 2/30
593/593 ━━━━━━━━━━━━━━━━━━━━ 150s 254ms/step - accuracy: 0.6642 - loss: 0.9809 - val_accuracy: 0.7113 - val_loss: 0.9284
Epoch 3/30
593/593 ━━━━━━━━━━━━━━━━━━━━ 542s 915ms/step - accuracy: 0.7125 - loss: 0.8417 - val_accuracy: 0.7081 - val_loss: 0.8403
Epoch 4/30
593/593 ━━━━━━━━━━━━━━━━━━━━ 192s 245ms/step - accuracy: 0.7352 - loss: 0.7523 - val_accuracy: 0.7741 - val_loss: 0.7288
Epoch 5/30
593/593 ━━━━━━━━━━━━━━━━━━━━ 140s 235ms/step - accuracy: 0.7471 - loss: 0.6924 - val_accuracy: 0.6634 - val_loss: 0.7522
Epoch 6/30
593/593 ━━━━━━━━━━━━━━━━━━━━ 150s 253ms/step - accuracy: 0.7608 - loss: 0.6281 - val_accuracy: 0.7822 - val_loss: 0.6180
Epoch 7/30
593/593 ━━━━━━━━━━━━━━━━━━━━ 198s 333ms/step - accuracy: 0.7692 - loss: 0.6132 - val_accuracy: 0.8203 - val_loss: 0.5974
Epoch 8/30
593/593 ━━━━━━━━━━━━━━━━━━━━ 176s 297ms/step - accuracy: 0.7840 -

In [15]:
model.save("age_model.keras")